# Luna 1 — Dataset Preparation (ChatML conversion)

This notebook is **stage 1** of the Luna fine-tuning pipeline: pulling every topic-file dataset from the HuggingFace Hub and converting it into Qwen2.5-Coder's ChatML training format. The same dataset feeds both **Luna 1 Lite** (base: Qwen2.5-Coder-3B) and **Luna 1** (base: Qwen2.5-Coder-7B) — neither model gets a different slice of data for now, so there's one shared conversion pipeline here.

**What this notebook does today:**
- Installs dependencies and imports everything up front
- Lists and downloads every `.parquet` topic file from the dataset repo on the Hub (no need to hardcode filenames — new topic batches are picked up automatically)
- Converts each Parquet file directly into ChatML using `render_to_chatml.py`, with no intermediate JSONL file on disk
- Validates the output and shows a rendered sample
- Saves the ChatML dataset locally, and optionally pushes it to a HuggingFace dataset repo so it persists across Colab runtimes instead of living only in `/content`

**What this notebook does NOT do:**
- Download a base Qwen2.5-Coder checkpoint or run the actual fine-tune. Both belong to the separate fine-tuning notebook (`02_train_luna.ipynb`, not yet written) — dataset prep and training are kept as two independent notebooks, each downloading only what it needs.

> Why keep raw and ChatML separate instead of just storing ChatML on the Hub?
> The raw schema (`plan`, individual `tool_calls`, `meta.efficiency_label`, ...) is the actual source of truth we edit and extend. Parquet is a columnar mirror of that same raw schema — not a different format. ChatML is a *rendering* of that data for one specific tokenizer/template. Doing the conversion here, in a notebook, means there's never a stale/out-of-sync ChatML copy sitting on the Hub — you always render fresh from source right before training. The optional push in section 8 is about persistence across runtimes, not about making ChatML a second source of truth: it's still a re-renderable cache, just one that survives a Colab restart.

## 1. Setup

Install the libraries this notebook needs.

In [ ]:
!pip install -q datasets huggingface_hub pandas pyarrow numpy

## 2. Imports

All imports for this notebook live in one place, so it's easy to see the full dependency surface at a glance instead of hunting through later cells.

In [ ]:
import json
import os
import sys

import numpy as np
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download

## 3. Download dataset

Config for the dataset repo, then a full listing of every `.parquet` topic file currently in it — new topic batches (e.g. `web_backend.parquet`, `tools.parquet`, and anything added later) are picked up automatically, nothing to hardcode here.

In [ ]:
# --- Config: edit these for your repo/paths ---
HF_REPO_ID = "sinamsv00/Luna"   # HF dataset repo (source of truth for the raw data)
LOCAL_DIR = "/content/luna_dataset"

os.makedirs(LOCAL_DIR, exist_ok=True)

In [ ]:
# List every file in the dataset repo, keep only the Parquet topic files.
api = HfApi()
all_files = api.list_repo_files(repo_id=HF_REPO_ID, repo_type="dataset")
parquet_files = sorted(f for f in all_files if f.endswith(".parquet"))

print(f"Found {len(parquet_files)} topic file(s) in {HF_REPO_ID}:")
for f in parquet_files:
    print(" -", f)

In [ ]:
# Download each Parquet topic file, plus the conversion script.
parquet_paths = [
    hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=filename,
        repo_type="dataset",
        local_dir=LOCAL_DIR,
    )
    for filename in parquet_files
]

script_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename="render_to_chatml.py",
    repo_type="dataset",
    local_dir=LOCAL_DIR,
)

print("Downloaded topic files:")
for p in parquet_paths:
    print(" -", p)
print("Conversion script:", script_path)

In [ ]:
sys.path.insert(0, LOCAL_DIR)

from render_to_chatml import render_record

print("Loaded render_record() from render_to_chatml.py")

## 4. Convert Parquet → ChatML

Each Parquet file mirrors the raw JSONL schema exactly (`plan`, `tool_calls`, `meta`, ...) — it's a columnar copy of the same records, not a different format. So each row is read straight into a `dict` and handed to `render_record()` directly, with **no intermediate JSONL file** written to disk.

One wrinkle: `DataFrame.to_dict()` round-trips nested list columns (`messages`, `tool_calls`) as `numpy.ndarray` instead of native Python lists, which trips up `render_record()`'s truthiness checks. `_denumpify()` below walks each record and converts those back to plain `list`/`dict` before rendering.

In [ ]:
def _denumpify(obj):
    """Recursively convert numpy arrays (and numpy scalars) back to plain Python.

    pandas.DataFrame.to_dict() round-trips any column that was a nested
    list (messages, tool_calls, ...) as a numpy.ndarray instead of a
    native list, which breaks render_record()'s `or []` / truthiness
    checks. This walks the structure and restores plain dict/list/scalar
    types so render_record() sees exactly the same shape it would from
    the original JSONL.

    Args:
        obj: Any value coming out of DataFrame.to_dict(orient="records").

    Returns:
        The same structure with all numpy.ndarray/np.generic replaced by
        native list/scalar equivalents.
    """
    if isinstance(obj, np.ndarray):
        return [_denumpify(item) for item in obj.tolist()]
    if isinstance(obj, dict):
        return {k: _denumpify(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_denumpify(item) for item in obj]
    if isinstance(obj, np.generic):
        return obj.item()
    return obj


def convert_parquet_to_chatml(parquet_path: str, out_path: str) -> int:
    """Render every record in a raw-schema Parquet file straight into ChatML.

    Reads the Parquet file, converts each row back into the same dict shape
    used by the raw JSONL schema (denumpifying nested list columns along
    the way), and renders it with render_record() — no intermediate JSONL
    file is written.

    Args:
        parquet_path: Path to a topic file's raw-schema .parquet file.
        out_path: Where to write the resulting ChatML .jsonl file.

    Returns:
        The number of records converted.
    """
    df = pd.read_parquet(parquet_path)
    records = [_denumpify(rec) for rec in df.to_dict(orient="records")]

    with open(out_path, "w", encoding="utf-8") as f:
        for rec in records:
            text = render_record(rec)
            f.write(
                json.dumps({"text": text, "meta": rec.get("meta", {})}, ensure_ascii=False)
                + "\n"
            )

    return len(records)

In [ ]:
# Convert every downloaded topic file, tracking the output paths for the sanity checks below.
chatml_paths = []

for parquet_path in parquet_paths:
    out_name = os.path.basename(parquet_path).replace(".parquet", "_chatml.jsonl")
    out_path = os.path.join(LOCAL_DIR, out_name)
    n = convert_parquet_to_chatml(parquet_path, out_path)
    chatml_paths.append(out_path)
    print(f"{os.path.basename(parquet_path)}: {n} records -> {out_path}")

## 5. Sanity-check the output

In [ ]:
all_records = []
for path in chatml_paths:
    with open(path, encoding="utf-8") as f:
        all_records.extend(json.loads(l) for l in f if l.strip())

print(f"{len(all_records)} total records converted across {len(chatml_paths)} topic file(s).\n")

# Show one full rendered example
sample = all_records[0]
print("=" * 80)
print(f"efficiency_label: {sample['meta']['efficiency_label']}  |  tool_calls: {sample['meta']['num_tool_calls']}")
print("=" * 80)
print(sample["text"])

In [ ]:
# Quick structural checks before this is considered training-ready
required_tags_present = all(
    "<|im_start|>" in rec["text"] and "<|im_end|>" in rec["text"]
    for rec in all_records
)
print("All records contain ChatML tags:", required_tags_present)

# Rough token-length sanity check (character count as a cheap proxy pre-tokenizer)
lengths = [len(rec["text"]) for rec in all_records]
print(f"Char length -- min: {min(lengths)}, max: {max(lengths)}, avg: {sum(lengths)//len(lengths)}")

## 6. Verify against the actual Qwen2.5-Coder tokenizer

`render_to_chatml.py` renders `tool` messages as `user`-role turns wrapped in `<tool_response>` tags, since this is the common convention for chat templates without a dedicated `tool` role. **Verify this matches the real `tokenizer_config.json`** for whichever Qwen2.5-Coder checkpoint you're targeting before training — some checkpoints do support a native tool role.

This dataset feeds two checkpoints (Luna 1 Lite and Luna 1) that share the same tokenizer family, so either one is representative here — pick whichever you're about to train next by uncommenting it below.

In [ ]:
from transformers import AutoTokenizer

# Pick the checkpoint you're about to train next -- both share the Qwen2.5-Coder
# tokenizer family, so either is representative for this notebook's sanity checks.
MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"   # Luna 1 Lite (small)
# MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"  # Luna 1 (large)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Does this checkpoint's chat template define a native 'tool' role?
has_native_tool_role = "tool" in (tokenizer.chat_template or "")
print(f"{MODEL_ID} chat_template mentions a native tool role:", has_native_tool_role)
print()
print("If True: consider remapping our <tool_response>-wrapped user turns to")
print("the native tool role before training, for template fidelity.")
print("If False: our current user-role wrapping convention is a safe default.")

In [ ]:
# Tokenize one sample end-to-end as a final sanity pass
tokens = tokenizer(sample["text"])["input_ids"]
print(f"Sample record tokenizes to {len(tokens)} tokens.")
print("Decoded back (first 300 chars):")
print(tokenizer.decode(tokens)[:300])

## 7. Save locally

In [ ]:
# Each topic file's ChatML output is already saved under LOCAL_DIR as it was
# converted in section 4. Listed here as a single confirmation point before
# the optional Hub push below.
for p in chatml_paths:
    size_kb = os.path.getsize(p) / 1024
    print(f"{p}  ({size_kb:.1f} KB)")

## 8. Push ChatML to the Hub *(optional)*

Colab's local disk (`/content`) doesn't survive a runtime reset, and this notebook is deliberately separate from the fine-tuning notebook — so without this step, every fresh training session would have to re-download the Parquet files and re-run the conversion in section 4 from scratch.

Set `PUSH_TO_HUB = True` and point `CHATML_REPO_ID` at a dataset repo you control to persist the converted ChatML files there instead. **One shared destination repo is used regardless of which model trains next** (Luna 1 Lite or Luna 1) — both currently train on the exact same dataset, so there's no need to duplicate the ChatML output per model. If that changes later (e.g. Luna 1 gets extra data Lite doesn't), this section will need a per-model repo or subfolder instead.

This still doesn't make ChatML a second source of truth — it's a cache of a specific render, kept alongside the raw dataset for convenience. If the raw Parquet files change, re-run this notebook and push again.

In [ ]:
# --- Config: flip this on and set your own repo to persist ChatML output ---
PUSH_TO_HUB = False   # <-- set True to enable the push below
CHATML_REPO_ID = "YOUR_USERNAME/luna-chatml"   # <-- your own HF dataset repo for ChatML output

In [ ]:
if PUSH_TO_HUB:
    api.create_repo(repo_id=CHATML_REPO_ID, repo_type="dataset", exist_ok=True)

    for chatml_path in chatml_paths:
        api.upload_file(
            path_or_fileobj=chatml_path,
            path_in_repo=os.path.basename(chatml_path),
            repo_id=CHATML_REPO_ID,
            repo_type="dataset",
        )
        print("Uploaded:", chatml_path, "->", CHATML_REPO_ID)
else:
    print("PUSH_TO_HUB is False -- skipping upload. ChatML files remain local to this runtime only.")